# Ghost Architecture & Detection Pipeline

Spiking Ghost convolutions and the SDDetect->DFLDecode->Dist2BBox->NMS pipeline.

In [ ]:
import numpy as np
from talon import ir

## 1. SGhostConv

In [ ]:
gc = ir.SGhostConv(
    primary_weight=np.random.randn(16, 8, 1, 1).astype(np.float32)*0.1,
    primary_bias=np.zeros(16, dtype=np.float32),
    cheap_weight=np.random.randn(16, 1, 3, 3).astype(np.float32)*0.1,
    cheap_bias=np.zeros(16, dtype=np.float32),
)
print(f"Primary: {gc.primary_weight.shape}, Cheap: {gc.cheap_weight.shape}")

## 2. GhostBasicBlock1 (Stride-2)

In [ ]:
in_ch, cp0, cp_r, cp2 = 32, 16, 8, 8
gb1 = ir.GhostBasicBlock1(
    cvres_primary_weight=np.random.randn(cp_r, in_ch, 1, 1).astype(np.float32)*0.1,
    cvres_primary_bias=np.zeros(cp_r, dtype=np.float32),
    cvres_cheap_weight=np.random.randn(cp_r, 1, 3, 3).astype(np.float32)*0.1,
    cvres_cheap_bias=np.zeros(cp_r, dtype=np.float32),
    cv0_primary_weight=np.random.randn(cp0, in_ch, 1, 1).astype(np.float32)*0.1,
    cv0_primary_bias=np.zeros(cp0, dtype=np.float32),
    cv0_cheap_weight=np.random.randn(cp0, 1, 3, 3).astype(np.float32)*0.1,
    cv0_cheap_bias=np.zeros(cp0, dtype=np.float32),
    cv2_primary_weight=np.random.randn(cp2, cp0, 1, 1).astype(np.float32)*0.1,
    cv2_primary_bias=np.zeros(cp2, dtype=np.float32),
    cv2_cheap_weight=np.random.randn(cp2, 1, 3, 3).astype(np.float32)*0.1,
    cv2_cheap_bias=np.zeros(cp2, dtype=np.float32),
    stride=2,
)
print(f"GB1: cv0={cp0}, cvres={cp_r}, cv2={cp2}, stride={gb1.stride}")

## 3. GhostBasicBlock2 (No Stride)

In [ ]:
in_ch, cp0, cp2 = 32, 16, 8
gb2 = ir.GhostBasicBlock2(
    cv0_primary_weight=np.random.randn(cp0, in_ch, 1, 1).astype(np.float32)*0.1,
    cv0_primary_bias=np.zeros(cp0, dtype=np.float32),
    cv0_cheap_weight=np.random.randn(cp0, 1, 3, 3).astype(np.float32)*0.1,
    cv0_cheap_bias=np.zeros(cp0, dtype=np.float32),
    cv2_primary_weight=np.random.randn(cp2, cp0, 1, 1).astype(np.float32)*0.1,
    cv2_primary_bias=np.zeros(cp2, dtype=np.float32),
    cv2_cheap_weight=np.random.randn(cp2, 1, 3, 3).astype(np.float32)*0.1,
    cv2_cheap_bias=np.zeros(cp2, dtype=np.float32),
)
print(f"GB2: cv0={cp0}, cv2={cp2}")

## 4. SGhostEncoderLite

In [ ]:
enc = ir.SGhostEncoderLite(
    conv1_weight=np.random.randn(16, 3, 3, 3).astype(np.float32)*0.1,
    conv1_bias=np.zeros(16, dtype=np.float32),
    ghost_primary_weight=np.random.randn(16, 16, 1, 1).astype(np.float32)*0.1,
    ghost_primary_bias=np.zeros(16, dtype=np.float32),
    ghost_cheap_weight=np.random.randn(16, 1, 3, 3).astype(np.float32)*0.1,
    ghost_cheap_bias=np.zeros(16, dtype=np.float32),
    stride=2,
)
print(f"Encoder: conv1={enc.conv1_weight.shape}, stride={enc.stride}")

## 5. Detection Pipeline

In [ ]:
det = ir.SDDetect(
    num_classes=80, reg_max=16, stride=8,
    cv2_w0=np.random.randn(64,32,3,3).astype(np.float32)*0.01, cv2_b0=np.zeros(64, dtype=np.float32),
    cv2_w1=np.random.randn(64,64,3,3).astype(np.float32)*0.01, cv2_b1=np.zeros(64, dtype=np.float32),
    cv2_w2=np.random.randn(64,64,1,1).astype(np.float32)*0.01, cv2_b2=np.zeros(64, dtype=np.float32),
    cv3_w0=np.random.randn(80,32,3,3).astype(np.float32)*0.01, cv3_b0=np.zeros(80, dtype=np.float32),
    cv3_w1=np.random.randn(80,80,3,3).astype(np.float32)*0.01, cv3_b1=np.zeros(80, dtype=np.float32),
    cv3_w2=np.random.randn(80,80,1,1).astype(np.float32)*0.01, cv3_b2=np.zeros(80, dtype=np.float32),
)
dfl = ir.DFLDecode(reg_max=16)
d2b = ir.Dist2BBox(stride=8)
nms = ir.NMS(iou_threshold=0.45, score_threshold=0.25, max_detections=300)

dn = {"feat": ir.Input(np.array([32,80,80])), "det": det, "dfl": dfl, "d2b": d2b, "nms": nms, "out": ir.Output(np.array([300,6]))}
de = [("feat","det"),("det","dfl"),("dfl","d2b"),("d2b","nms"),("nms","out")]
dg = ir.Graph(nodes=dn, edges=de)
print(f"Detection: {len(dg.nodes)} nodes, DAG={dg.is_dag}")